# 🖌️ PixArt-Sigma LoRA — Chinese Ink Wash Paintings

> **End-to-end fine-tuning and inference notebook** for domain-adapting PixArt-Sigma (DiT) to the Chinese Ink Wash Painting style using LoRA.  
> Dataset: [`jianwenzhao/ink-wash` (trainB)](https://www.kaggle.com/datasets/jianwenzhao/ink-wash/data?select=trainB) from Kaggle.  
> Based on the [Project Proposal](Project-Proposal.md) and [README](README.md).

---

## 📋 Pipeline Overview

```
Phase 1: Data          Phase 2: Training      Phase 3: Inference     Phase 4: Evaluation
──────────────────     ──────────────────     ──────────────────     ──────────────────
📥 Download Kaggle     📐 3 LoRA Ranks        ⏱️ 4 Step configs      🤖 CLIP Score
   trainB split           (4, 8, 16)              (5, 10, 20, 50)        (OpenCLIP)
✂️ Nested Subsets      📈 3 Data Scales       🎯 3 Guidance Scales   👥 Human Ratings
   50 / 100 / 300         (50 / 100 / 300)        (3.0 / 5.0 / 7.5)     (1–5 scale)
🏷️ BLIP Captioning    💾 9 LoRA .safetensors  ✍️ 3 Prompt Types      🗺️ Pareto Frontier
   + trigger phrase
```

| Phase | What it does | Key Output |
|-------|-------------|------------|
| **1. Data** | Download `trainB`, split, BLIP-caption + trigger phrase | 3 nested image-caption datasets |
| **2. Training** | LoRA at ranks 4, 8, 16 × 3 data scales | 9 `.safetensors` LoRA weights |
| **3. Inference** | Grid: steps × guidance × prompts | 324 generated ink-wash images |
| **4. Evaluation** | CLIP score, latency, Pareto frontier | Analysis plots & summary CSV |

> **Resolution:** 512 × 512 (reduced memory; per project proposal).  
> **Base model:** `PixArt-alpha/PixArt-Sigma-XL-2-512-MS`.

---
## ⚙️ 0. Environment Setup

In [ ]:
# Install core dependencies
!pip install -q \
    torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install -q \
    diffusers[torch] \
    transformers \
    accelerate \
    safetensors \
    peft \
    datasets \
    Pillow \
    tqdm \
    pandas \
    matplotlib \
    seaborn \
    scipy \
    open-clip-torch \
    sentencepiece \
    kaggle                   # Kaggle API for downloading the ink-wash dataset

print("✅ All dependencies installed.")

In [ ]:
import os, sys, time, csv, shutil, random, itertools, json, math
from pathlib import Path
from datetime import datetime

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected — training will be very slow on CPU.")

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 🔧 GLOBAL CONFIGURATION
# ─────────────────────────────────────────────────────────────────

CFG = {
    # === Kaggle dataset ===
    "KAGGLE_DATASET"  : "jianwenzhao/ink-wash",      # Kaggle dataset slug
    "KAGGLE_SPLIT"    : "trainB",                     # Sub-folder to use
    "RAW_DATA_DIR"    : "./data/raw/trainB",          # Where Kaggle downloads land

    # === Processed data paths ===
    "DATA_ROOT"       : "./data",
    "DATASET_SIZES"   : [50, 100, 300],

    # === Model ===
    # 512-px variant — lower VRAM requirement (fits ~12 GB)
    "BASE_MODEL"      : "PixArt-alpha/PixArt-Sigma-XL-2-512-MS",
    "RESOLUTION"      : 512,

    # === Domain trigger phrase (appended to every caption) ===
    "TRIGGER_PHRASE"  : "traditional Chinese ink wash painting style, shuimo hua",

    # === Training hyperparameters ===
    "LORA_RANKS"      : [4, 8, 16],
    "MAX_TRAIN_STEPS" : 1000,
    "TRAIN_BATCH_SIZE": 1,
    "LEARNING_RATE"   : 1e-4,
    "CHECKPOINTING_STEPS": 500,
    "SEED"            : 42,
    "MIXED_PRECISION" : "fp16",

    # === Inference grid ===
    "STEPS_LIST"      : [5, 10, 20, 50],
    "GUIDANCE_LIST"   : [3.0, 5.0, 7.5],

    # === Output paths ===
    "OUTPUT_ROOT"     : "./outputs",
    "INFERENCE_DIR"   : "./inference_results",
    "METRICS_CSV"     : "./metrics.csv",
}

CFG["DATASET_DIRS"] = [f"dataset_{n}" for n in CFG["DATASET_SIZES"]]

for d in [CFG["DATA_ROOT"], CFG["OUTPUT_ROOT"], CFG["INFERENCE_DIR"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

SUPPORTED_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

def scan_images(folder):
    """Return sorted list of image Paths in a folder."""
    return sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in SUPPORTED_EXTS)

print("✅ Configuration ready:")
for k, v in CFG.items():
    print(f"   {k:<25} = {v}")

---
## 💾 Phase 1: Data — Ink Wash `trainB` from Kaggle

The dataset is `jianwenzhao/ink-wash` on Kaggle (~90 MB zip).  
We use the **`trainB`** split — the *ink wash painting* side of the unpaired image-to-image dataset.

### Dataset Facts
- **Source:** Kaggle — [jianwenzhao/ink-wash](https://www.kaggle.com/datasets/jianwenzhao/ink-wash)
- **Split used:** `trainB` (ink-wash paintings)
- **License:** Unknown (verify before commercial use)

### 1.1 Download via Kaggle API

**Prerequisites:** Place your `kaggle.json` credentials file at `~/.kaggle/kaggle.json` (Linux/Mac) or `%USERPROFILE%\.kaggle\kaggle.json` (Windows).  
Get it from [Kaggle → Account → API → Create New Token](https://www.kaggle.com/settings/account).

In [ ]:
import subprocess

raw_dir  = Path(CFG["RAW_DATA_DIR"]).parent.parent   # ./data/raw
raw_dir.mkdir(parents=True, exist_ok=True)

trainB_dir = Path(CFG["RAW_DATA_DIR"])   # ./data/raw/trainB

if trainB_dir.exists() and len(scan_images(trainB_dir)) > 0:
    print(f"✅ trainB already downloaded ({len(scan_images(trainB_dir))} images at {trainB_dir})")
else:
    print("📥 Downloading ink-wash dataset from Kaggle...")
    result = subprocess.run(
        [
            "kaggle", "datasets", "download",
            "-d", CFG["KAGGLE_DATASET"],
            "--path", str(raw_dir),
            "--unzip",
        ],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("❌ Kaggle download failed:")
        print(result.stderr)
        print("\n📌 Manual alternative:")
        print("   1. Go to https://www.kaggle.com/datasets/jianwenzhao/ink-wash")
        print("   2. Download and unzip the dataset")
        print(f"  3. Place the 'trainB' folder at: {trainB_dir.resolve()}")
    else:
        imgs = scan_images(trainB_dir)
        print(f"\n✅ Downloaded! trainB contains {len(imgs)} images at {trainB_dir}")

In [ ]:
# Inspect the downloaded trainB images
trainB_dir = Path(CFG["RAW_DATA_DIR"])
all_images = scan_images(trainB_dir) if trainB_dir.exists() else []

print(f"📁 trainB path   : {trainB_dir.resolve()}")
print(f"🖼️  Images found  : {len(all_images)}")

if len(all_images) == 0:
    print("\n⚠️  No images found. See download instructions above.")
else:
    # Show a sample grid of 8 ink wash images
    sample = all_images[:8]
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, p in zip(axes.flat, sample):
        ax.imshow(Image.open(p).convert("RGB"))
        ax.set_title(p.name, fontsize=7)
        ax.axis("off")
    plt.suptitle("Sample Images — Ink Wash trainB", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # Image size distribution
    sizes = [Image.open(p).size for p in tqdm(all_images[:50], desc="Checking sizes")]
    ws, hs = zip(*sizes)
    print(f"\nImage width  : min={min(ws)}, max={max(ws)}, median={int(np.median(ws))}")
    print(f"Image height : min={min(hs)}, max={max(hs)}, median={int(np.median(hs))}")

### 1.2 Quality Filtering & Deduplication

In [ ]:
def filter_images(
    image_paths: list,
    min_size: int = 200,       # Minimum width and height in pixels
    min_file_bytes: int = 5000, # Minimum file size (avoids tiny/corrupt files)
) -> list:
    """
    Filter out images that are too small, corrupt, or grayscale-only.
    Returns a list of valid image Paths.
    """
    valid = []
    skipped = {"too_small": 0, "corrupt": 0, "tiny_file": 0}

    for p in tqdm(image_paths, desc="Filtering images"):
        if p.stat().st_size < min_file_bytes:
            skipped["tiny_file"] += 1
            continue
        try:
            img = Image.open(p)
            w, h = img.size
            if w < min_size or h < min_size:
                skipped["too_small"] += 1
                continue
            valid.append(p)
        except Exception:
            skipped["corrupt"] += 1

    print(f"\n✅ Valid images  : {len(valid)}/{len(image_paths)}")
    print(f"   Skipped       : {skipped}")
    return valid


if all_images:
    filtered_images = filter_images(all_images)
    # Shuffle deterministically so subsets are random but reproducible
    random.seed(CFG["SEED"])
    random.shuffle(filtered_images)
    print(f"\n🎲 Shuffled {len(filtered_images)} images with seed={CFG['SEED']}")
else:
    filtered_images = []
    print("⚠️  No images to filter.")

### 1.3 Create Nested Dataset Subsets

Splits the filtered pool into strictly nested subsets: **50 ⊆ 100 ⊆ 300** images (or fewer if the dataset is smaller than 300).

In [ ]:
def create_nested_subsets(source_images, sizes, dataset_dirs, data_root):
    """Copy images into nested subset directories and stub empty .txt caption files."""
    data_root = Path(data_root)
    for size, dir_name in zip(sizes, dataset_dirs):
        subset_dir = data_root / dir_name
        subset_dir.mkdir(parents=True, exist_ok=True)
        subset_imgs = source_images[:size]

        for src in tqdm(subset_imgs, desc=f"Copying {dir_name}", leave=False):
            dst     = subset_dir / src.name
            txt_dst = subset_dir / (src.stem + ".txt")
            if not dst.exists():
                shutil.copy2(src, dst)
            if not txt_dst.exists():
                txt_dst.write_text("", encoding="utf-8")

        print(f"   ✅ {dir_name:<18} {len(subset_imgs):>4} images  →  {subset_dir}")


if filtered_images:
    # Warn if we have fewer than 300 images
    available = len(filtered_images)
    actual_sizes = [min(s, available) for s in CFG["DATASET_SIZES"]]
    if available < 300:
        print(f"⚠️  Only {available} images available — adjusting subset sizes to {actual_sizes}")

    print("✂️  Creating nested subsets...")
    create_nested_subsets(
        source_images = filtered_images,
        sizes         = actual_sizes,
        dataset_dirs  = CFG["DATASET_DIRS"],
        data_root     = CFG["DATA_ROOT"],
    )
    print("\n✅ Subset creation complete!")
else:
    print("⚠️  Skipping — no source images available.")

### 1.4 Auto-Captioning with BLIP

Uses `Salesforce/blip-image-captioning-base` (as specified in the proposal) to generate one caption per image, then appends the domain trigger phrase `"traditional Chinese ink wash painting style, shuimo hua"` for consistent style conditioning.

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

def load_blip_model(device="cuda"):
    """Load BLIP captioning model (Salesforce/blip-image-captioning-base)."""
    print("Loading BLIP captioning model...")
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained(
        "Salesforce/blip-image-captioning-base",
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    ).to(device)
    model.eval()
    print("✅ BLIP model loaded.")
    return processor, model


def caption_single_image(image, processor, model, device, trigger_phrase=""):
    """Generate a caption for one PIL image and append the trigger phrase."""
    dtype = torch.float16 if device == "cuda" else torch.float32
    inputs = processor(images=image, return_tensors="pt").to(device)
    inputs = {k: v.to(dtype) if v.dtype == torch.float32 else v for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=80, num_beams=4, early_stopping=True)
    caption = processor.decode(out[0], skip_special_tokens=True).strip()
    if trigger_phrase:
        caption = f"{caption}, {trigger_phrase}"
    return caption


def auto_caption_dataset(dataset_dir, processor, model, device, trigger_phrase="", overwrite=False):
    """Caption all images in a directory. Skips existing non-empty captions."""
    dataset_dir = Path(dataset_dir)
    images = scan_images(dataset_dir)
    captioned, skipped = 0, 0

    for img_path in tqdm(images, desc=f"Captioning {dataset_dir.name}"):
        txt_path = dataset_dir / (img_path.stem + ".txt")
        if txt_path.exists() and txt_path.stat().st_size > 0 and not overwrite:
            skipped += 1
            continue
        image   = Image.open(img_path).convert("RGB")
        caption = caption_single_image(image, processor, model, device, trigger_phrase)
        txt_path.write_text(caption, encoding="utf-8")
        captioned += 1

    print(f"   ✅ {dataset_dir.name}: {captioned} new captions, {skipped} already present.")


print("✅ BLIP captioning utilities defined.")

In [ ]:
# Run auto-captioning on dataset_300 (largest subset).
# Captions are then propagated to dataset_50 and dataset_100.

# ⚠️ First run downloads ~940 MB of BLIP weights.

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
largest_dir = Path(CFG["DATA_ROOT"]) / CFG["DATASET_DIRS"][-1]   # dataset_300

if largest_dir.exists() and len(scan_images(largest_dir)) > 0:
    blip_processor, blip_model = load_blip_model(DEVICE)

    # Caption the full set
    auto_caption_dataset(
        dataset_dir    = largest_dir,
        processor      = blip_processor,
        model          = blip_model,
        device         = DEVICE,
        trigger_phrase = CFG["TRIGGER_PHRASE"],
        overwrite      = False,   # Set True to regenerate all
    )

    # Propagate captions to smaller subsets
    for dir_name in CFG["DATASET_DIRS"][:-1]:   # dataset_50, dataset_100
        subset_dir = Path(CFG["DATA_ROOT"]) / dir_name
        for img in scan_images(subset_dir):
            src_txt = largest_dir / (img.stem + ".txt")
            dst_txt = subset_dir  / (img.stem + ".txt")
            if src_txt.exists() and (not dst_txt.exists() or dst_txt.stat().st_size == 0):
                shutil.copy2(src_txt, dst_txt)
        print(f"   ✅ Propagated captions → {dir_name}")

    del blip_model, blip_processor
    torch.cuda.empty_cache()
    print("\n🧹 BLIP unloaded from GPU.")
else:
    print("⚠️  Skipping captioning — no images in dataset_300.")

In [ ]:
# Preview a few generated captions
print("📝 Sample captions from dataset_300:\n")
d300 = Path(CFG["DATA_ROOT"]) / "dataset_300"
txt_files = [p for p in d300.iterdir() if p.suffix == ".txt" and p.stat().st_size > 0]
for t in sorted(txt_files)[:5]:
    print(f"  [{t.stem}]  {t.read_text(encoding='utf-8')[:120]}")

print()
print("📊 Dataset verification:")
print("─" * 55)
for dir_name in CFG["DATASET_DIRS"]:
    d = Path(CFG["DATA_ROOT"]) / dir_name
    if not d.exists():
        print(f"  ❌ {dir_name}: not found")
        continue
    imgs  = scan_images(d)
    txts  = [p for p in d.iterdir() if p.suffix == ".txt"]
    empty = [t for t in txts if t.stat().st_size == 0]
    ok    = "✅" if imgs and not empty else "⚠️ "
    print(f"  {ok} {dir_name:<18} images={len(imgs):>4}  captions={len(txts):>4}  empty={len(empty)}")

---
## ⚙️ Phase 2: Multi-Configuration LoRA Training

Train **9 LoRA checkpoints** across the full hyperparameter matrix:

| | `rank=4` | `rank=8` | `rank=16` |
|---|---|---|---|
| `dataset_50`  | ✓ | ✓ | ✓ |
| `dataset_100` | ✓ | ✓ | ✓ |
| `dataset_300` | ✓ | ✓ | ✓ |

**Model:** `PixArt-alpha/PixArt-Sigma-XL-2-512-MS`  
**LoRA targets:** DiT Transformer attention projections (`to_q`, `to_k`, `to_v`, `to_out`, feed-forward layers)

### 2.1 Dataset Class & LoRA Config

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from peft import LoraConfig, get_peft_model


class InkWashDataset(Dataset):
    """Image-caption dataset for PixArt-Sigma LoRA fine-tuning on ink wash images."""

    def __init__(self, data_dir: str, resolution: int = 512):
        self.data_dir    = Path(data_dir)
        self.resolution  = resolution
        self.image_paths = scan_images(self.data_dir)
        if not self.image_paths:
            raise ValueError(f"No images found in {data_dir}")

        self.transform = transforms.Compose([
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.LANCZOS),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        txt_path = img_path.with_suffix(".txt")
        image    = Image.open(img_path).convert("RGB")
        caption  = txt_path.read_text(encoding="utf-8").strip() if txt_path.exists() else ""
        return {"pixel_values": self.transform(image), "caption": caption}


def get_lora_config(rank: int) -> LoraConfig:
    """LoRA config targeting PixArt-Sigma DiT attention + FFN layers."""
    return LoraConfig(
        r            = rank,
        lora_alpha   = rank * 2,
        target_modules = [
            "to_q", "to_k", "to_v", "to_out.0",
            "proj_in", "proj_out",
            "ff.net.0.proj", "ff.net.2",
        ],
        lora_dropout = 0.05,
        bias         = "none",
    )


print("✅ InkWashDataset and LoRA config defined.")

### 2.2 `train_lora()` — Single Training Run

In [ ]:
from accelerate import Accelerator
from accelerate.utils import set_seed
from diffusers import PixArtSigmaPipeline


def train_lora(
    base_model_id, data_dir, output_dir,
    rank, resolution=512, max_train_steps=1000,
    train_batch_size=1, learning_rate=1e-4,
    checkpointing_steps=500, mixed_precision="fp16", seed=42,
):
    """
    Fine-tune PixArt-Sigma with LoRA on the ink wash domain.
    Returns a dict of training stats.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    accelerator = Accelerator(
        mixed_precision=mixed_precision,
        gradient_accumulation_steps=4,
    )
    set_seed(seed)
    device = accelerator.device
    dtype  = torch.float16 if mixed_precision == "fp16" else torch.float32

    # ── Load base model components ────────────────────────────────
    print(f"  📦 Loading {base_model_id} ...")
    pipe        = PixArtSigmaPipeline.from_pretrained(base_model_id, torch_dtype=dtype)
    vae         = pipe.vae.to(device)
    text_encoder= pipe.text_encoder.to(device)
    transformer = pipe.transformer.to(device)
    tokenizer   = pipe.tokenizer
    noise_sched = pipe.scheduler

    # Freeze all except LoRA
    vae.requires_grad_(False)
    text_encoder.requires_grad_(False)
    transformer.requires_grad_(False)

    # ── Apply LoRA ────────────────────────────────────────────────
    transformer = get_peft_model(transformer, get_lora_config(rank))
    transformer.print_trainable_parameters()

    # ── Data ──────────────────────────────────────────────────────
    dataset    = InkWashDataset(data_dir, resolution=resolution)
    dataloader = DataLoader(
        dataset, batch_size=train_batch_size,
        shuffle=True, num_workers=2, pin_memory=True,
    )

    # ── Optimizer + LR scheduler ──────────────────────────────────
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, transformer.parameters()),
        lr=learning_rate, betas=(0.9, 0.999), weight_decay=1e-2,
    )
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max_train_steps
    )

    transformer, optimizer, dataloader, lr_scheduler = accelerator.prepare(
        transformer, optimizer, dataloader, lr_scheduler
    )

    # ── Training loop ─────────────────────────────────────────────
    global_step, running_loss, loss_history = 0, 0.0, []
    num_epochs = math.ceil(max_train_steps / len(dataloader))
    start_time = time.time()

    print(f"  🚀 Training {max_train_steps} steps | {num_epochs} epochs | rank={rank}")

    for epoch in range(num_epochs):
        transformer.train()
        for batch in dataloader:
            if global_step >= max_train_steps:
                break

            pixel_values = batch["pixel_values"].to(device, dtype=dtype)
            captions     = batch["caption"]

            with accelerator.accumulate(transformer):
                with torch.no_grad():
                    latents = vae.encode(pixel_values).latent_dist.sample()
                    latents = latents * vae.config.scaling_factor

                    text_inputs = tokenizer(
                        captions, padding="max_length",
                        max_length=tokenizer.model_max_length,
                        truncation=True, return_tensors="pt",
                    ).to(device)
                    encoder_hidden_states = text_encoder(**text_inputs).last_hidden_state

                noise     = torch.randn_like(latents)
                timesteps = torch.randint(
                    0, noise_sched.config.num_train_timesteps,
                    (latents.shape[0],), device=device
                ).long()
                noisy_latents = noise_sched.add_noise(latents, noise, timesteps)

                model_pred = transformer(
                    noisy_latents,
                    encoder_hidden_states=encoder_hidden_states,
                    timestep=timesteps,
                    return_dict=False,
                )[0]

                loss = torch.nn.functional.mse_loss(model_pred, noise, reduction="mean")
                accelerator.backward(loss)

                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(transformer.parameters(), 1.0)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            running_loss += loss.item()
            loss_history.append(loss.item())
            global_step  += 1

            if global_step % checkpointing_steps == 0:
                ckpt = output_dir / f"checkpoint-{global_step}"
                accelerator.unwrap_model(transformer).save_pretrained(ckpt)
                print(f"    💾 Checkpoint @ step {global_step}")

            if global_step >= max_train_steps:
                break

    # Save final LoRA
    accelerator.unwrap_model(transformer).save_pretrained(output_dir)
    elapsed  = time.time() - start_time
    avg_loss = running_loss / max(global_step, 1)
    print(f"  ✅ Done in {elapsed/60:.1f} min | avg loss = {avg_loss:.4f} → {output_dir}")

    return {
        "rank": rank, "data_dir": str(data_dir), "output_dir": str(output_dir),
        "steps": global_step, "avg_loss": avg_loss,
        "elapsed_s": elapsed, "loss_history": loss_history,
    }


print("✅ train_lora() defined.")

### 2.3 Training Matrix Loop (9 Configurations)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Set RUN_TRAINING = True to launch training.
# Each run skips automatically if the checkpoint already exists.
# Estimated time: 15–30 min per run on A100 → ~2–5 hrs total.
# ─────────────────────────────────────────────────────────────────

RUN_TRAINING = False   # ← flip to True when ready

training_results = []

if RUN_TRAINING:
    combos = list(itertools.product(CFG["LORA_RANKS"], CFG["DATASET_DIRS"]))
    for idx, (rank, dir_name) in enumerate(combos, 1):
        run_label   = f"lora_r{rank}_{dir_name}"
        output_path = Path(CFG["OUTPUT_ROOT"]) / run_label
        data_path   = Path(CFG["DATA_ROOT"])   / dir_name

        print(f"\n{'='*60}")
        print(f"🚀  Run {idx}/{len(combos)}: {run_label}")
        print(f"{'='*60}")

        if (output_path / "pytorch_lora_weights.safetensors").exists():
            print(f"   ⏩ Already trained — skipping.")
            training_results.append({"run": run_label, "status": "skipped"})
            continue

        if not data_path.exists() or not scan_images(data_path):
            print(f"   ❌ No data at {data_path} — skipping.")
            training_results.append({"run": run_label, "status": "no_data"})
            continue

        result = train_lora(
            base_model_id       = CFG["BASE_MODEL"],
            data_dir            = str(data_path),
            output_dir          = str(output_path),
            rank                = rank,
            resolution          = CFG["RESOLUTION"],
            max_train_steps     = CFG["MAX_TRAIN_STEPS"],
            train_batch_size    = CFG["TRAIN_BATCH_SIZE"],
            learning_rate       = CFG["LEARNING_RATE"],
            checkpointing_steps = CFG["CHECKPOINTING_STEPS"],
            mixed_precision     = CFG["MIXED_PRECISION"],
            seed                = CFG["SEED"],
        )
        result["run"] = run_label
        result["status"] = "completed"
        training_results.append(result)
        torch.cuda.empty_cache()

    summary_df = pd.DataFrame(training_results)
    summary_path = Path(CFG["OUTPUT_ROOT"]) / "training_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"\n✅ Training matrix complete!")
    print(f"📊 Summary saved → {summary_path}")
    display(summary_df[["run", "status", "avg_loss", "elapsed_s", "steps"]])
else:
    print("⏸️  Training skipped (RUN_TRAINING=False).")
    print("   Set RUN_TRAINING = True and re-run to start.")

### 2.4 Loss Curve Visualization

In [ ]:
def plot_loss_curves(results):
    completed = [r for r in results if r.get("status") == "completed" and "loss_history" in r]
    if not completed:
        print("No completed runs to plot.")
        return

    ncols = min(3, len(completed))
    nrows = math.ceil(len(completed) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharey=True)
    axes = np.array(axes).flatten()

    colors = plt.cm.plasma(np.linspace(0.15, 0.85, len(completed)))
    for ax, result, color in zip(axes, completed, colors):
        losses   = result["loss_history"]
        window   = max(1, len(losses) // 20)
        smoothed = pd.Series(losses).rolling(window, min_periods=1).mean()
        ax.plot(losses, alpha=0.2, color=color, linewidth=0.8)
        ax.plot(smoothed, color=color, linewidth=2)
        ax.set_title(result["run"], fontsize=9, fontweight="bold")
        ax.set_xlabel("Step"); ax.set_ylabel("MSE Loss")
        ax.grid(True, alpha=0.3); ax.spines[["top", "right"]].set_visible(False)

    for ax in axes[len(completed):]:
        ax.set_visible(False)

    plt.suptitle("LoRA Training Loss Curves — Ink Wash Domain",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    out = Path(CFG["OUTPUT_ROOT"]) / "training_loss_curves.png"
    plt.savefig(out, bbox_inches="tight", dpi=150)
    plt.show()
    print(f"📈 Loss curves saved → {out}")


if training_results:
    plot_loss_curves(training_results)
else:
    print("Run Phase 2.3 first.")

---
## 🔮 Phase 3: Automated Grid Inference

Generate images across the evaluation grid:
- **9 LoRA weights** × **4 step configs** × **3 guidance scales** × **3 prompts** = **324 images**
- Ink-wash–specific prompts of escalating complexity
- Inference **latency (ms)** recorded per image for Phase 4

> ⚠️ Do not generate manually — use the loop below.

### 3.1 Ink Wash Evaluation Prompts

In [ ]:
# Prompts of escalating complexity — ink wash domain
# All end with the domain trigger phrase for style consistency.

TRIGGER = CFG["TRIGGER_PHRASE"]

PROMPTS = {
    "simple": (
        f"A mountain, {TRIGGER}."
    ),
    "combo": (
        f"A misty mountain range with pine trees reflected in a calm river, {TRIGGER}."
    ),
    "complex": (
        f"A solitary scholar seated in a bamboo pavilion perched on a dramatic cliff, "
        f"overlooking layered mountain peaks shrouded in morning mist, "
        f"sparse brushwork, expressive ink gradients, {TRIGGER}."
    ),
}

print("🎯 Evaluation prompts:")
for name, text in PROMPTS.items():
    print(f"\n  [{name}]\n   {text}")

### 3.2 Grid Generation Loop

In [ ]:
# Set RUN_INFERENCE = True after training is complete.

RUN_INFERENCE = False   # ← flip to True

if RUN_INFERENCE:
    from diffusers import PixArtSigmaPipeline

    inference_dir = Path(CFG["INFERENCE_DIR"])
    inference_dir.mkdir(parents=True, exist_ok=True)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    metrics_rows = []

    print(f"📦 Loading base pipeline: {CFG['BASE_MODEL']}")
    pipe = PixArtSigmaPipeline.from_pretrained(
        CFG["BASE_MODEL"], torch_dtype=torch.float16, use_safetensors=True,
    ).to(DEVICE)
    pipe.set_progress_bar_config(disable=True)

    total = (len(CFG["LORA_RANKS"]) * len(CFG["DATASET_DIRS"]) *
             len(CFG["STEPS_LIST"]) * len(CFG["GUIDANCE_LIST"]) * len(PROMPTS))
    print(f"\n📊 Total images: {total}")

    pbar = tqdm(total=total, desc="Grid inference")

    for rank, dir_name in itertools.product(CFG["LORA_RANKS"], CFG["DATASET_DIRS"]):
        lora_path = Path(CFG["OUTPUT_ROOT"]) / f"lora_r{rank}_{dir_name}"

        if not lora_path.exists():
            skip_n = len(CFG["STEPS_LIST"]) * len(CFG["GUIDANCE_LIST"]) * len(PROMPTS)
            print(f"\n  ⚠️  LoRA missing: {lora_path.name} — skipping {skip_n} images.")
            pbar.update(skip_n)
            continue

        pipe.load_lora_weights(str(lora_path))

        for steps, g_scale, (p_name, prompt_text) in itertools.product(
            CFG["STEPS_LIST"], CFG["GUIDANCE_LIST"], PROMPTS.items()
        ):
            fname     = f"r{rank}_{dir_name}_step{steps}_g{g_scale}_{p_name}.png"
            save_path = inference_dir / fname

            if save_path.exists():
                pbar.update(1)
                continue

            generator = torch.Generator(DEVICE).manual_seed(CFG["SEED"])
            t0 = time.perf_counter()
            image = pipe(
                prompt              = prompt_text,
                num_inference_steps = steps,
                guidance_scale      = g_scale,
                generator           = generator,
                height              = CFG["RESOLUTION"],
                width               = CFG["RESOLUTION"],
            ).images[0]
            latency_ms = (time.perf_counter() - t0) * 1000

            image.save(save_path)
            metrics_rows.append({
                "filename"   : fname,
                "rank"       : rank,
                "dataset"    : dir_name,
                "steps"      : steps,
                "guidance"   : g_scale,
                "prompt_type": p_name,
                "prompt_text": prompt_text,
                "latency_ms" : round(latency_ms, 1),
                "image_path" : str(save_path),
            })
            pbar.update(1)

        pipe.unload_lora_weights()
        torch.cuda.empty_cache()

    pbar.close()

    df_inf = pd.DataFrame(metrics_rows)
    df_inf.to_csv(CFG["METRICS_CSV"], index=False)
    print(f"\n✅ Inference complete! {len(metrics_rows)} images generated.")
    print(f"📊 Latency log saved → {CFG['METRICS_CSV']}")
    display(df_inf.head(8))

else:
    print("⏸️  Inference skipped (RUN_INFERENCE=False).")
    print("   Set RUN_INFERENCE = True after training.")

### 3.3 Preview Generated Image Grid

In [ ]:
def preview_grid(inference_dir, filter_rank=8, filter_dataset="dataset_300", filter_prompt="complex"):
    """
    Preview a slice of the grid.
    Rows = guidance scales (3, 5, 7.5), Columns = step counts (5, 10, 20, 50).
    """
    inference_dir = Path(inference_dir)
    step_vals     = sorted(CFG["STEPS_LIST"])
    guide_vals    = sorted(CFG["GUIDANCE_LIST"])

    fig, axes = plt.subplots(len(guide_vals), len(step_vals),
                             figsize=(3.5 * len(step_vals), 3.5 * len(guide_vals)))

    for r, g in enumerate(guide_vals):
        for c, s in enumerate(step_vals):
            ax    = axes[r][c]
            fname = f"r{filter_rank}_{filter_dataset}_step{s}_g{g}_{filter_prompt}.png"
            fpath = inference_dir / fname
            if fpath.exists():
                ax.imshow(Image.open(fpath).convert("RGB"))
            else:
                ax.text(0.5, 0.5, "Not generated", ha="center", va="center",
                        transform=ax.transAxes, fontsize=8)
                ax.set_facecolor("#f5f0eb")
            ax.set_title(f"steps={s} | cfg={g}", fontsize=8)
            ax.axis("off")

    plt.suptitle(
        f"Ink Wash Grid — LoRA r{filter_rank} | {filter_dataset} | '{filter_prompt}' prompt\n"
        "Rows: guidance scale  |  Cols: inference steps",
        fontsize=11, fontweight="bold", y=1.01
    )
    plt.tight_layout()
    out = inference_dir / f"preview_r{filter_rank}_{filter_dataset}_{filter_prompt}.png"
    plt.savefig(out, bbox_inches="tight", dpi=120)
    plt.show()
    print(f"🖼️  Grid preview saved → {out}")


preview_grid(
    CFG["INFERENCE_DIR"],
    filter_rank    = 8,
    filter_dataset = "dataset_300",
    filter_prompt  = "complex",
)

---
## 📊 Phase 4: Evaluation & Analysis

Two evaluation tracks as specified in the proposal:

| Track | Method | Metric |
|-------|--------|--------|
| **Quantitative** | OpenCLIP `ViT-L-14` | CLIPScore (text-image alignment) |
| **Quantitative** | Wall-clock timer | Inference latency (ms) |
| **Qualitative** | Human blind rating | Style alignment + structural integrity (1–5) |
| **Summary** | Pareto frontier | Quality vs. speed trade-off |


### 4.1 CLIPScore Computation

In [ ]:
import open_clip


def load_clip_model(device="cuda"):
    print("Loading OpenCLIP ViT-L-14 (openai weights)...")
    model, _, preprocess = open_clip.create_model_and_transforms("ViT-L-14", pretrained="openai")
    model = model.to(device).eval()
    tokenizer = open_clip.get_tokenizer("ViT-L-14")
    print("✅ CLIP model loaded.")
    return model, preprocess, tokenizer


def compute_clip_score(model, preprocess, tokenizer, image, prompt, device):
    img_t   = preprocess(image).unsqueeze(0).to(device)
    txt_tok = tokenizer([prompt]).to(device)
    with torch.no_grad():
        img_f = model.encode_image(img_t)
        txt_f = model.encode_text(txt_tok)
        img_f /= img_f.norm(dim=-1, keepdim=True)
        txt_f /= txt_f.norm(dim=-1, keepdim=True)
    return round((img_f @ txt_f.T).item(), 6)


print("✅ CLIP score helpers defined.")

In [ ]:
# Compute CLIP scores for all generated images.
# Adds a 'clip_score' column to metrics.csv.

RUN_CLIP_EVAL = False   # ← Set True after inference

if RUN_CLIP_EVAL:
    metrics_csv = Path(CFG["METRICS_CSV"])
    if not metrics_csv.exists():
        print("❌ metrics.csv not found. Run Phase 3 first.")
    else:
        df = pd.read_csv(metrics_csv)
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        clip_model, clip_pre, clip_tok = load_clip_model(DEVICE)

        scores = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc="CLIPScore"):
            img_path = Path(row["image_path"])
            if not img_path.exists():
                scores.append(None); continue
            img = Image.open(img_path).convert("RGB")
            scores.append(compute_clip_score(clip_model, clip_pre, clip_tok,
                                             img, row["prompt_text"], DEVICE))

        df["clip_score"] = scores
        df.to_csv(metrics_csv, index=False)
        del clip_model; torch.cuda.empty_cache()

        print(f"\n✅ CLIPScores computed for {df['clip_score'].notna().sum()} images.")
        print(df.groupby(["rank", "dataset"])["clip_score"].mean().unstack().to_string())
else:
    print("⏸️  CLIP eval skipped (RUN_CLIP_EVAL=False).")

### 4.2 Load Results (or Synthetic Demo Data)

In [ ]:
metrics_csv = Path(CFG["METRICS_CSV"])

if metrics_csv.exists() and "clip_score" in pd.read_csv(metrics_csv).columns:
    df = pd.read_csv(metrics_csv)
    print(f"✅ Loaded {len(df)} rows with CLIP scores from {metrics_csv}")
    display(df.describe())
else:
    # ── Synthetic demo data ───────────────────────────────────────
    print("⚠️  Generating synthetic demo data (ink wash domain characteristics)...")
    np.random.seed(CFG["SEED"])
    rows = []
    for rank in CFG["LORA_RANKS"]:
        for dataset in CFG["DATASET_DIRS"]:
            data_bonus = {"dataset_50": 0.0, "dataset_100": 0.015, "dataset_300": 0.032}[dataset]
            rank_bonus = {4: 0.0, 8: 0.008, 16: 0.013}[rank]
            for steps in CFG["STEPS_LIST"]:
                for guidance in CFG["GUIDANCE_LIST"]:
                    for p_name in PROMPTS:
                        # Ink wash: quality saturates quickly (artistic style != photorealism)
                        step_bonus  = np.log1p(steps) * 0.012
                        cfg_bonus   = (guidance - 3.0) / 18
                        prompt_pen  = {"simple": 0.0, "combo": -0.005, "complex": -0.012}[p_name]
                        clip_score  = min(0.36, max(0.14,
                            0.21 + data_bonus + rank_bonus + step_bonus + cfg_bonus + prompt_pen
                            + np.random.normal(0, 0.004)
                        ))
                        latency_ms  = steps * (1.8 + rank * 0.04 + np.random.normal(0, 0.15)) * 1000
                        rows.append({
                            "filename"   : f"r{rank}_{dataset}_step{steps}_g{guidance}_{p_name}.png",
                            "rank"       : rank,
                            "dataset"    : dataset,
                            "steps"      : steps,
                            "guidance"   : guidance,
                            "prompt_type": p_name,
                            "prompt_text": PROMPTS[p_name],
                            "latency_ms" : round(max(latency_ms, 200), 1),
                            "clip_score" : round(clip_score, 6),
                        })
    df = pd.DataFrame(rows)
    df.to_csv(metrics_csv, index=False)
    print(f"📊 Synthetic demo data ({len(df)} rows) saved → {metrics_csv}")
    display(df.head(10))

### 4.3 Heatmaps: CLIP Score & Latency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── CLIP Score heatmap: rank × dataset ───────────────────────────
pivot_clip = df.groupby(["rank", "dataset"])["clip_score"].mean().unstack()
sns.heatmap(pivot_clip, ax=axes[0], annot=True, fmt=".3f",
            cmap="YlGnBu", linewidths=0.5,
            cbar_kws={"label": "Avg CLIPScore"})
axes[0].set_title("Mean CLIPScore\nLoRA Rank × Dataset Size",
                  fontsize=13, fontweight="bold")
axes[0].set_xlabel("Dataset")
axes[0].set_ylabel("LoRA Rank")

# ── Latency heatmap: steps × guidance ────────────────────────────
pivot_lat = df.groupby(["steps", "guidance"])["latency_ms"].mean().unstack()
sns.heatmap(pivot_lat, ax=axes[1], annot=True, fmt=".0f",
            cmap="OrRd", linewidths=0.5,
            cbar_kws={"label": "Avg Latency (ms)"})
axes[1].set_title("Mean Inference Latency (ms)\nSteps × Guidance Scale",
                  fontsize=13, fontweight="bold")
axes[1].set_xlabel("Guidance Scale")
axes[1].set_ylabel("Inference Steps")

plt.suptitle("Ink Wash LoRA — Evaluation Heatmaps", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(Path(CFG["OUTPUT_ROOT"]) / "heatmaps.png", bbox_inches="tight", dpi=150)
plt.show()

### 4.4 CLIPScore by Prompt Complexity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
prompt_order = ["simple", "combo", "complex"]
colors = {"dataset_50": "#6b8cba", "dataset_100": "#e07b39", "dataset_300": "#4a9e6b"}

for idx, p_type in enumerate(prompt_order):
    sub   = df[df["prompt_type"] == p_type]
    pivot = sub.groupby(["rank", "dataset"])["clip_score"].mean().reset_index()
    for dataset, grp in pivot.groupby("dataset"):
        axes[idx].plot(
            grp["rank"], grp["clip_score"],
            marker="o", linewidth=2.2,
            color=colors.get(dataset, "grey"),
            label=dataset,
        )
    axes[idx].set_title(f"'{p_type}' prompt", fontsize=11, fontweight="bold")
    axes[idx].set_xlabel("LoRA Rank")
    axes[idx].set_ylabel("Avg CLIPScore" if idx == 0 else "")
    axes[idx].set_xticks([4, 8, 16])
    axes[idx].grid(True, alpha=0.3)
    axes[idx].spines[["top", "right"]].set_visible(False)
    if idx == 2:
        axes[idx].legend(title="Dataset", fontsize=9)

plt.suptitle("CLIPScore vs LoRA Rank\nby Prompt Complexity — Ink Wash Domain",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(Path(CFG["OUTPUT_ROOT"]) / "clip_by_prompt.png", bbox_inches="tight", dpi=150)
plt.show()

### 4.5 🗺️ Pareto Frontier — Quality vs. Inference Speed

The **Pareto frontier** connects configurations where no other setting achieves **both** better quality **and** lower latency simultaneously. Points on the frontier are the practical deployment sweet spots.

In [ ]:
def compute_pareto(df, x_col, y_col):
    """Extract Pareto-optimal points (minimize x, maximize y)."""
    df = df.dropna(subset=[x_col, y_col]).sort_values(x_col).copy()
    pareto, best_y = [], -np.inf
    for _, row in df.iterrows():
        if row[y_col] > best_y:
            pareto.append(row)
            best_y = row[y_col]
    return pd.DataFrame(pareto)


def plot_pareto(df, prompt_filter="complex", x_col="latency_ms", y_col="clip_score"):
    sub = df[df["prompt_type"] == prompt_filter].copy()
    agg = (
        sub.groupby(["rank", "dataset", "steps", "guidance"])[[x_col, y_col]]
        .mean().reset_index()
    )

    pareto = compute_pareto(agg, x_col, y_col)

    rank_colors = {4: "#4e79a7", 8: "#f28e2b", 16: "#59a14f"}
    step_sizes  = {5: 45, 10: 90, 20: 140, 50: 210}

    fig, ax = plt.subplots(figsize=(13, 7))

    for rank, grp in agg.groupby("rank"):
        ax.scatter(
            grp[x_col], grp[y_col],
            s       = grp["steps"].map(step_sizes).fillna(90),
            alpha   = 0.50,
            color   = rank_colors[rank],
            label   = f"LoRA rank={rank}",
            edgecolors="white", linewidths=0.6,
        )

    if len(pareto) >= 2:
        ax.plot(
            pareto[x_col], pareto[y_col],
            color="crimson", linewidth=2.5, linestyle="--",
            marker="D", markersize=9, markerfacecolor="crimson",
            zorder=5, label="Pareto Frontier",
        )
        for _, row in pareto.iterrows():
            ax.annotate(
                f"r{int(row['rank'])} | s{int(row['steps'])}\n{row['dataset'].split('_')[1]} imgs",
                xy=(row[x_col], row[y_col]),
                xytext=(10, 6), textcoords="offset points",
                fontsize=7.5, color="crimson",
                arrowprops=dict(arrowstyle="->", color="crimson", lw=0.8),
            )

    step_handles = [
        plt.scatter([], [], s=step_sizes[s], color="grey", alpha=0.6, label=f"{s} steps")
        for s in sorted(step_sizes)
    ]
    rank_leg = ax.legend(loc="lower right", fontsize=9, title="LoRA Rank", framealpha=0.9)
    ax.add_artist(rank_leg)
    ax.legend(handles=step_handles, loc="upper left", fontsize=9,
              title="Inference Steps", framealpha=0.9)

    ax.set_xlabel("Inference Latency (ms)",  fontsize=12)
    ax.set_ylabel("CLIPScore  ↑ better",      fontsize=12)
    ax.set_title(
        f"Quality vs Latency — Pareto Frontier\n"
        f"Ink Wash LoRA | '{prompt_filter}' prompt",
        fontsize=14, fontweight="bold"
    )
    ax.grid(True, alpha=0.22)
    ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    out = Path(CFG["OUTPUT_ROOT"]) / f"pareto_{prompt_filter}.png"
    plt.savefig(out, bbox_inches="tight", dpi=180)
    plt.show()
    print(f"🗺️  Pareto plot saved → {out}")

    print("\n🏆 Pareto-Optimal Configurations:")
    display(
        pareto[["rank", "dataset", "steps", "guidance", x_col, y_col]]
        .rename(columns={x_col: "latency_ms", y_col: "clip_score"})
        .sort_values("latency_ms").reset_index(drop=True)
    )


for ptype in ["simple", "combo", "complex"]:
    plot_pareto(df, prompt_filter=ptype)

### 4.6 Full Experiment Summary

In [ ]:
summary = (
    df.groupby(["rank", "dataset", "steps", "guidance"])
    .agg(
        mean_clip   = ("clip_score",  "mean"),
        std_clip    = ("clip_score",  "std"),
        mean_lat_ms = ("latency_ms",  "mean"),
        n_images    = ("filename",    "count"),
    )
    .reset_index()
    .sort_values("mean_clip", ascending=False)
)
summary["quality_per_second"] = summary["mean_clip"] / (summary["mean_lat_ms"] / 1000)

print("📊 Top 20 configurations by CLIPScore:")
display(
    summary.head(20).style
    .background_gradient(cmap="YlGn", subset=["mean_clip"])
    .background_gradient(cmap="OrRd_r", subset=["mean_lat_ms"])
    .format({"mean_clip": "{:.4f}", "mean_lat_ms": "{:.0f} ms",
             "quality_per_second": "{:.5f}"})
)

out_summary = Path(CFG["OUTPUT_ROOT"]) / "experiment_summary.csv"
summary.to_csv(out_summary, index=False)
print(f"\n✅ Full summary saved → {out_summary}")

### 4.7 LoRA Strength (Scale) Ablation

Investigates how the **LoRA weight scale** (`lora_scale`) controls style strength at inference time — distinct from the training rank. This supports the proposal's goal of studying prompt and LoRA-weight interpolation.

In [ ]:
# Run LoRA scale sweep for a single checkpoint.
# Vary lora_scale from 0.0 (base model) to 1.5 (over-stylised).

RUN_SCALE_ABLATION = False   # ← Set True after training

SCALE_VALUES     = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
ABLATION_RANK    = 8
ABLATION_DATASET = "dataset_300"
ABLATION_PROMPT  = PROMPTS["complex"]
ABLATION_STEPS   = 20
ABLATION_CFG     = 5.0

if RUN_SCALE_ABLATION:
    from diffusers import PixArtSigmaPipeline

    lora_path = Path(CFG["OUTPUT_ROOT"]) / f"lora_r{ABLATION_RANK}_{ABLATION_DATASET}"
    DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

    if not lora_path.exists():
        print(f"❌ LoRA not found: {lora_path}")
    else:
        pipe = PixArtSigmaPipeline.from_pretrained(
            CFG["BASE_MODEL"], torch_dtype=torch.float16,
        ).to(DEVICE)
        pipe.load_lora_weights(str(lora_path))
        pipe.set_progress_bar_config(disable=True)

        ablation_images = []
        for scale in tqdm(SCALE_VALUES, desc="LoRA scale sweep"):
            pipe.set_adapters(["default"], adapter_weights=[scale])
            gen = torch.Generator(DEVICE).manual_seed(CFG["SEED"])
            img = pipe(
                ABLATION_PROMPT,
                num_inference_steps=ABLATION_STEPS,
                guidance_scale=ABLATION_CFG,
                generator=gen,
                height=CFG["RESOLUTION"], width=CFG["RESOLUTION"],
            ).images[0]
            ablation_images.append((scale, img))

        # Display
        n = len(ablation_images)
        fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 4))
        for ax, (scale, img) in zip(axes, ablation_images):
            ax.imshow(img); ax.set_title(f"scale={scale}", fontsize=9); ax.axis("off")
        plt.suptitle("LoRA Strength Ablation — Ink Wash Style",
                     fontsize=12, fontweight="bold")
        plt.tight_layout()
        out = Path(CFG["OUTPUT_ROOT"]) / "lora_scale_ablation.png"
        plt.savefig(out, bbox_inches="tight", dpi=150)
        plt.show()
        print(f"🎛️  Scale ablation saved → {out}")

        del pipe; torch.cuda.empty_cache()
else:
    print("⏸️  Scale ablation skipped (RUN_SCALE_ABLATION=False).")

### 4.8 Human Evaluation Template Export

In [ ]:
def export_human_eval_template(df, output_csv, prompt_filter="complex", n_samples=30, seed=42):
    """
    Export a shuffled CSV for blind human evaluation of style alignment
    and structural integrity (1–5 scales).
    """
    sub = df[df["prompt_type"] == prompt_filter].copy()
    if len(sub) > n_samples:
        sub = sub.sample(n_samples, random_state=seed)
    sub = sub.sample(frac=1, random_state=seed).reset_index(drop=True)
    sub["eval_id"]              = range(1, len(sub) + 1)
    sub["style_alignment"]      = ""   # Rater fills 1–5
    sub["structural_integrity"] = ""   # Rater fills 1–5
    sub["notes"]                = ""

    cols = ["eval_id", "image_path", "prompt_text",
            "style_alignment", "structural_integrity", "notes"]
    export = sub[[c for c in cols if c in sub.columns]]
    export.to_csv(output_csv, index=False)

    print(f"📋 Human eval template ({len(export)} images) → {output_csv}")
    print()
    print("Rating rubric:")
    print("  Style Alignment (1–5)")
    print("    1 = Generic / no ink wash feel")
    print("    3 = Partial style match")
    print("    5 = Perfect ink wash aesthetic")
    print()
    print("  Structural Integrity (1–5)")
    print("    1 = Chaotic brushwork / broken composition")
    print("    3 = Acceptable but imprecise")
    print("    5 = Clean, expressive, well-structured lines")


eval_out = Path(CFG["OUTPUT_ROOT"]) / "human_eval_template.csv"
export_human_eval_template(df, eval_out, prompt_filter="complex", n_samples=30)

---
## 📁 Final Project File Structure

```
efficient-pixart-sigma-lora/
│
├── data/
│   ├── raw/trainB/              ← Downloaded Kaggle ink-wash images
│   ├── dataset_50/              ← 50 images + BLIP captions
│   ├── dataset_100/             ← 100 images + BLIP captions
│   └── dataset_300/             ← 300 images + BLIP captions
│
├── outputs/
│   ├── lora_r4_dataset_50/      ← .safetensors LoRA weights
│   ├── lora_r4_dataset_100/
│   ├── lora_r4_dataset_300/
│   ├── lora_r8_dataset_50/
│   ├── lora_r8_dataset_100/
│   ├── lora_r8_dataset_300/
│   ├── lora_r16_dataset_50/
│   ├── lora_r16_dataset_100/
│   ├── lora_r16_dataset_300/
│   ├── training_summary.csv
│   ├── training_loss_curves.png
│   ├── heatmaps.png
│   ├── clip_by_prompt.png
│   ├── pareto_complex.png
│   ├── lora_scale_ablation.png
│   ├── experiment_summary.csv
│   └── human_eval_template.csv
│
├── inference_results/           ← 324 ink-wash generated images
│   └── r8_dataset_300_step20_g5.0_complex.png  (example)
│
└── metrics.csv                  ← latency + CLIPScore for all 324 images
```

---

## 🏁 Quick-Start Checklist

| # | Step | Cell |
|---|------|------|
| 1 | Install dependencies | §0 |
| 2 | Place `kaggle.json` credentials | manual |
| 3 | Download `trainB` ink-wash images | §1.1 |
| 4 | Filter & shuffle images | §1.2 |
| 5 | Create nested subsets (50/100/300) | §1.3 |
| 6 | BLIP auto-caption + trigger phrase | §1.4 |
| 7 | Set `RUN_TRAINING = True` | §2.3 |
| 8 | Monitor loss curves | §2.4 |
| 9 | Set `RUN_INFERENCE = True` | §3.2 |
| 10 | Preview image grid | §3.3 |
| 11 | Set `RUN_CLIP_EVAL = True` | §4.1 |
| 12 | Analyse heatmaps | §4.3 |
| 13 | Plot Pareto frontier | §4.5 |
| 14 | LoRA scale ablation | §4.7 |
| 15 | Export human eval template | §4.8 |

---
*Dataset: [jianwenzhao/ink-wash (trainB)](https://www.kaggle.com/datasets/jianwenzhao/ink-wash/data?select=trainB) · Model: `PixArt-alpha/PixArt-Sigma-XL-2-512-MS` · See [Project-Proposal.md](Project-Proposal.md) for full context.*